# Zbieranie korpusu kolokacji — Colab GPU

Parsuje polskie teksty Stanzą i zapisuje trójki kolokacyjne na Dysk Google.

**Zanim uruchomisz:** Środowisko wykonawcze → Zmień typ środowiska → **GPU (T4)**.

Kod pochodzi z repozytorium projektu, a nie jest wklejony do notebooka — celowo.
Ekstraktor trójek musi być **dokładnie ten sam**, którego użyje dodatek w czasie
pisania. Rozjazd logiki klucza między budową bazy a zapytaniem nie objawiłby się
żadnym błędem — tylko cicho zerową skutecznością podpowiedzi.

In [ ]:
# 1. Kontrola GPU — bez niego przebieg potrwa kilkanaście razy dłużej
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "BRAK GPU — zmień typ środowiska wykonawczego"

In [ ]:
# 2. Kod projektu + zależności
REPO = "https://github.com/SernikNaZimno/semantyczna-autokorekta.git"  # uzupełnij po założeniu repo

import os, shutil
if os.path.exists("projekt"):
    shutil.rmtree("projekt")
!git clone --depth 1 $REPO projekt
%cd projekt
!pip install -q stanza datasets

In [ ]:
# 3. Model polski (pobiera się raz, ~500 MB)
import stanza
stanza.download("pl", verbose=False)
print("model gotowy")

In [ ]:
# 4. Dysk Google — żeby wynik przeżył rozłączenie sesji
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
KATALOG = Path("/content/drive/MyDrive/kolokacje")
KATALOG.mkdir(parents=True, exist_ok=True)
print(KATALOG)

In [ ]:
# 5. Zbieranie
#
# Budżet jest PER ŹRÓDŁO. Bez podziału Wikipedia wypełniłaby cały limit —
# strumieniuje się szybciej niż C4, a chcemy oba rejestry.
#
# Przebieg próbny: 5 mln tokenów (~15 min na T4).
# Pełna skala: podnieś do 60/40 mln i licz kilka godzin.

import sys; sys.path.insert(0, ".")
from backend.pipeline import zbierz

BUDZET = {"wiki": 3_000_000, "web": 2_000_000}
WYJSCIE = KATALOG / "trojki_5M.tsv.gz"

stat = zbierz(BUDZET, WYJSCIE, gpu=True, partia=256)
stat

In [ ]:
# 6. Baza + pierwsze spojrzenie na sygnał
from backend.baza import BazaKolokacji, zbuduj
from backend.pipeline import czytaj_trojki

BAZA = KATALOG / "kolokacje_5M.sqlite"
print(zbuduj(czytaj_trojki(WYJSCIE), BAZA, min_pary=3))

with BazaKolokacji(BAZA) as db:
    print(db.statystyki())
    print(f"udział wiki {db.udzial_zrodla('wiki'):.1%}, web {db.udzial_zrodla('web'):.1%}")
    print()
    print("Czasowniki łączące się z 'decyzja' (obj:acc):")
    for k in db.alternatywy("obj:acc", "decyzja", limit=10):
        print(f"  {k.lemat:<16} f={k.f:<4} logDice={k.logdice:.2f}")
    print()
    f = db.czestosc_slotu_dep("obj:acc", "decyzja")
    print(f"częstość brzegowa 'decyzja' w obj:acc = {f}")
    print("slot zbadany" if db.slot_zbadany("obj:acc", "decyzja") else "slot NIEzbadany — silnik milczy")

In [ ]:
# 7. Kontrola skażenia domenowego
#
# Para o wysokiej częstości pochodząca wyłącznie z webu to zwykle boilerplate
# (stopka, klauzula, szablon ogłoszenia), a nie norma językowa.

with BazaKolokacji(BAZA) as db:
    print("Pary wyłącznie z webu, najczęstsze:")
    for h, s, d, f in db.con.execute(
        """SELECT p.head, p.slot, p.dep, p.f FROM pary p
           WHERE p.f >= 10 AND NOT EXISTS (
             SELECT 1 FROM pary_zrodlo z WHERE z.head=p.head AND z.slot=p.slot
             AND z.dep=p.dep AND z.zrodlo='wiki')
           ORDER BY p.f DESC LIMIT 20"""):
        print(f"  {h} --{s}--> {d}   f={f}")